# Week 39

In [ ]:
!pip install -q --upgrade transformers datasets sacrebleu rouge_score evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 16.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.


In [ ]:
import pandas as pd
import re
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import nltk
import transformers
from nltk.util import ngrams
import math
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk import FreqDist, ConditionalFreqDist
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch import nn
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader
from torch.optim import AdamW
from typing import List, Tuple
from transformers import MarianMTModel, MarianTokenizer, DistilBertTokenizerFast
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from tqdm import tqdm
import os
import evaluate
from transformers import Seq2SeqTrainingArguments
from sacrebleu.metrics import BLEU
from transformers import DataCollatorForSeq2Seq
from datasets import Dataset
from transformers import Seq2SeqTrainer
from google.colab import drive
drive.mount('/content/drive')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Mounted at /content/drive


In [ ]:
# Load the model
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
### NEEDED FOR TRANSLATION

# --- Device ---
# device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Load multilingual mBART50 model ---
# trans_model_name = "facebook/mbart-large-50-many-to-many-mmt"
# trans_tokenizer = AutoTokenizer.from_pretrained(trans_model_name)
# trans_model = AutoModelForSeq2SeqLM.from_pretrained(trans_model_name).to(device)

In [ ]:
LANG_MAP = {
    "te": "te_IN",
    "ar": "ar_AR",
    "ko": "ko_KR"
}
TARGET_LANG = "te_IN"

def translate_to_telugu_mbart(text, source_lang="en_XX"):
    tokenizer.src_lang = source_lang
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)

    translated_tokens = trans_model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id[TARGET_LANG],
        max_length=64,
        num_beams=5,
        early_stopping=True
    )

    translated_text = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
    return translated_text

# Path to save the translations
output_path = "/content/drive/MyDrive/translated_data/train_with_telugu.csv"

if os.path.exists(output_path):
    print(f"Found existing translations {output_path}")
    df_train_te = pd.read_csv(output_path)
else:
    print("No translations found, running translate_to_telugu_mbart")

    df_train_te = df_train_te[df_train_te['answer'].notnull()].reset_index(drop=True)

    tqdm.pandas(desc="Translating English answers to Telugu")
    df_train_te['answer_inlang_trans'] = df_train_te['answer'].progress_apply(
        lambda x: translate_to_telugu_mbart(x)
    )
    df_train_te['answer_inlang'] = df_train_te['answer_inlang'].combine_first(df_train_te['answer_inlang_trans'])
    df_train_te = df_train_te.drop(columns=['answer_inlang_trans'])

    df_train_te.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Translations saved to {output_path}")

val_output_path = "/content/drive/MyDrive/translated_data/val_with_telugu.csv"

if os.path.exists(val_output_path):
    print(f"Loading existing validation translations from {val_output_path}...")
    df_val_te = pd.read_csv(val_output_path)
else:
    print("Translating validation set...")
    tqdm.pandas(desc="Translating English answers to Telugu")
    df_val_te['answer_inlang_trans'] = df_val_te['answer'].progress_apply(
        lambda x: translate_to_telugu_mbart(x)
    )
    df_val_te['answer_inlang'] = df_val_te['answer_inlang'].combine_first(df_val_te['answer_inlang_trans'])
    df_val_te = df_val_te.drop(columns=['answer_inlang_trans'])
    df_val_te.to_csv(val_output_path, index=False, encoding="utf-8-sig")
    print(f"Saved translated validation set to {val_output_path}")

Found existing translations /content/drive/MyDrive/translated_data/train_with_telugu.csv
                                            question  \
0  ప్రపంచంలో  మొట్టమొదటి దూర విద్య విద్యాలయం ఏ దే...   
1      1959వ సంవత్సరంలో భారతదేశ ప్రధాన మంత్రి  ఎవరు?   
2  ఏ కాకతీయ రాజు కర్నూలు జిల్లాను చివరిగా పాలించాడు?   
3                                మానవ హక్కులు ఎన్ని?   
4       భారదేశంలో అత్యధిక జనాభా కలిగిన రాష్ట్రం ఏది?   

                                             context lang  answerable  \
0  Referred to as "People's University" by Charle...   te        True   
1  Since 1947, there have been 14 different prime...   te        True   
2  Rani Rudrama Devi (died 1289 or 1295), who def...   te        True   
3  The Declaration consists of 30 articles affirm...   te        True   
4  Uttar Pradesh (; IAST: "Uttar Pradeś" ) is a s...   te        True   

   answer_start            answer     answer_inlang  
0           236            London             లండన్  
1           220  Jawaharlal

## Question + Context

In [ ]:
# Get the question and context
def get_question_context(row):
    question = row['question']
    return question

df_train_te['input_text'] = df_train_te.apply(get_question_context, axis=1)
df_val_te['input_text'] = df_val_te.apply(get_question_context, axis=1)

df_train_te = df_train_te[df_train_te['answerable'] == True]

def get_target_language(row):
  answer_inlang = row['answer_inlang']
  return answer_inlang

def get_target_language_trans(row):
  answer_inlang_trans = row['answer_inlang_trans']
  return answer_inlang_trans

df_train_te['target_text'] = df_train_te.apply(get_target_language, axis=1)
df_val_te['target_text'] = df_val_te.apply(get_target_language_trans, axis=1)

In [ ]:
df_train_te = df_train_te[['input_text', 'target_text', 'answerable']]

df_val_te = df_val_te[['input_text', 'target_text', 'answerable']]

# Convert to HF Dataset
train_dataset = Dataset.from_pandas(df_train_te, preserve_index=False)
val_dataset = Dataset.from_pandas(df_val_te, preserve_index=False)

max_input_length = 768
max_target_length = 64

def tokenize_function(examples):
    model_inputs = tokenizer(examples["input_text"], max_length=max_input_length, truncation=True)
    labels = tokenizer(examples["target_text"], max_length=max_target_length, truncation=True)
    model_inputs["labels"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    return model_inputs

train_dataset = Dataset.from_pandas(df_train_te)
val_dataset = Dataset.from_pandas(df_val_te)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/1310 [00:00<?, ? examples/s]

Map:   0%|          | 0/384 [00:00<?, ? examples/s]

In [ ]:
bleu = BLEU()

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Load metrics
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
chrf_metric = evaluate.load("chrf")

def compute_metrics(eval_preds):
    predictions, labels = eval_preds

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = np.array(predictions)
    if predictions.ndim > 2:
        predictions = np.argmax(predictions, axis=-1)

    predictions = np.clip(predictions, 0, tokenizer.vocab_size - 1)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    preds_text = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels_text = tokenizer.batch_decode(labels, skip_special_tokens=True)

    em_scores = [int(pred.strip() == ref.strip()) for pred, ref in zip(preds_text, labels_text)]
    exact_match = np.mean(em_scores)

    bleu_score = bleu.corpus_score(preds_text, [labels_text]).score

    rouge_result = rouge_metric.compute(predictions=preds_text, references=labels_text)
    rouge_l = rouge_result["rougeL"]

    chrf_result = chrf_metric.compute(predictions=preds_text, references=labels_text)
    chrf_score = chrf_result["score"]

    return {
        "exact_match": exact_match,
        "bleu": bleu_score,
        "rougeL": rouge_l,
        "chrF": chrf_score
    }


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5-te-qa",
    label_smoothing_factor=0.1,
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=50,
    predict_with_generate=True,
    logging_dir="./logs",
    logging_steps=50,
    save_safetensors=False
)

In [ ]:
# Train the model
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

/tmp/ipython-input-999792941.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: melbyeandreas02 (melbyeandreas02-university-of-copenhagen) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
50,26.771100
100,22.635000
150,19.386700
200,18.740600
250,17.466300
300,15.943300
350,14.947200
400,13.573200
450,12.949000
500,11.702700


TrainOutput(global_step=8200, training_loss=7.014874278743092, metrics={'train_runtime': 3239.6875, 'train_samples_per_second': 20.218, 'train_steps_per_second': 2.531, 'total_flos': 1554203949772800.0, 'train_loss': 7.014874278743092, 'epoch': 50.0})

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

for example in tokenized_val.select(range(20)):
    input_text = example['input_text']
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512)

    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = model.generate(
    **inputs,
    max_length=64,
    num_beams=5,
    early_stopping=True,
    decoder_start_token_id=model.config.decoder_start_token_id)

    # print("Question + Context:", input_text)
    print("Generated Answer:", tokenizer.decode(outputs[0], skip_special_tokens=True))
    print("Reference:", example['target_text'])
    print("-----")

Generated Answer: మూడు
Reference: પોર્ટલેન્ડ
-----
Generated Answer: రెండు
Reference: இந்திய ઉપખંડ
-----
Generated Answer: Vivekanda
Reference: ఇင်္ဂလန်
-----
Generated Answer: íka
Reference: 1914
-----
Generated Answer: 1923
Reference: 28 జూలై 1914
-----
Generated Answer: రెండు
Reference: భారతదేశం
-----
Generated Answer: ఆరు
Reference: 122
-----
Generated Answer: 7 సంవత్సరాల
Reference: గురించి 5 లిటర్లు
-----
Generated Answer: చార్లెస్
Reference: JPMorgan చైస్ టవర్
-----
Generated Answer: ఉత్తర జూన్
Reference: 18
-----
Generated Answer: మూడు ઓક્ટોબર
Reference: 1926
-----
Generated Answer: అమెరికా
Reference: వెస్ట్ బెంగాల్
-----
Generated Answer: 15
Reference: బహుశా 10,000 మంది మాట్లాడారు 122
-----
Generated Answer: ఇంగ్లీష్
Reference: సెయింట్ పీటర్ బాబిజిక్
-----
Generated Answer: ఐదు
Reference: .eg
-----
Generated Answer: ఏడాన్
Reference: లాస్ అঞె জেলస్
-----
Generated Answer: ఇంగ్లీష్
Reference: Meghalaya, Mizoram, మరియు Nagaland
-----
Generated Answer: గ్రెర్
Reference: ఆఫ్రికా
---

In [ ]:
# Evaluate
results = trainer.evaluate()
print(results)

{'eval_loss': 6.078125, 'eval_exact_match': 0.0, 'eval_bleu': 0.20825601136345853, 'eval_rougeL': 0.0062499999999999995, 'eval_chrF': 2.240377048629137, 'eval_runtime': 6.7918, 'eval_samples_per_second': 56.539, 'eval_steps_per_second': 7.067, 'epoch': 50.0}


In [ ]:
# Evaluate on answerable and uanswerable
n_train = len(tokenized_train)
n_val = len(tokenized_val)

print(f"Training set size: {n_train}")
print(f"Validation set size: {n_val}")

n_val_answerable = sum(tokenized_val['answerable'])
n_val_unanswerable = n_val - n_val_answerable

print(f"Validation Answerable:   {n_val_answerable}")
print(f"Validation Unanswerable: {n_val_unanswerable}")

n_train_answerable = sum(tokenized_train['answerable'])
n_train_unanswerable = n_train - n_train_answerable

print(f"Training Answerable:   {n_train_answerable}")
print(f"Training Unanswerable: {n_train_unanswerable}")

val_answerable = tokenized_val.filter(lambda x: x["answerable"] == True)
val_unanswerable = tokenized_val.filter(lambda x: x["answerable"] == False)

results_answerable = trainer.evaluate(eval_dataset=val_answerable)
results_unanswerable = trainer.evaluate(eval_dataset=val_unanswerable)

print("Answerable:", results_answerable)
print("Unanswerable:", results_unanswerable)

Training set size: 1310
Validation set size: 384
Validation Answerable:   291
Validation Unanswerable: 93
Training Answerable:   1310
Training Unanswerable: 0


Filter:   0%|          | 0/384 [00:00<?, ? examples/s]

Filter:   0%|          | 0/384 [00:00<?, ? examples/s]

Answerable: {'eval_loss': 5.96942138671875, 'eval_exact_match': 0.0, 'eval_bleu': 0.21524896935099527, 'eval_rougeL': 0.006872852233676976, 'eval_chrF': 2.1356859737874294, 'eval_runtime': 5.0362, 'eval_samples_per_second': 57.781, 'eval_steps_per_second': 7.347, 'epoch': 50.0}
Unanswerable: {'eval_loss': 6.399578094482422, 'eval_exact_match': 0.0, 'eval_bleu': 0.0, 'eval_rougeL': 0.004301075268817204, 'eval_chrF': 2.590377005296612, 'eval_runtime': 1.5199, 'eval_samples_per_second': 61.189, 'eval_steps_per_second': 7.895, 'epoch': 50.0}
